# 03 — Predictive Modeling
**Auction Marketplace Segmentation & Price Intelligence Engine**

### Task: Price Intelligence Classification
Predict whether a listing is **underpriced**, **fair-priced**, or **overpriced** relative to comparable equipment.

Models evaluated: Logistic Regression (baseline), LightGBM (primary), XGBoost (comparison)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import json

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize

sns.set_theme(style='whitegrid')
print('Ready')

## 1. Load Data & Results

In [ ]:
df = pd.read_csv('../data/processed/listings_features.csv', parse_dates=['list_date'])

results_path = '../data/processed/model_results.json'
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    print("=== Model Comparison ===")
    for model_name in ['logistic_regression', 'lightgbm', 'xgboost']:
        m = results[model_name]
        print(f"  {m['model']:<25}  acc={m['accuracy']:.4f}  f1={m['f1_macro']:.4f}  auc={m['roc_auc']:.4f}")
    print(f"\nBest model: {results['best_model']}")
    print(f"Features used: {results['feature_count']}")
    print(f"Train/Test size: {results['train_size']:,} / {results['test_size']:,}")

## 2. Target Distribution

In [ ]:
print("Price label distribution:")
print(df['price_label'].value_counts())
print()
print("Class proportions:")
print(df['price_label'].value_counts(normalize=True).round(3))

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#e74c3c','#2ecc71','#e67e22']
df['price_label'].value_counts().plot.bar(ax=ax, color=colors, edgecolor='white', rot=0)
ax.set_title('Price Intelligence Target Distribution')
ax.set_ylabel('Count')
ax.set_xlabel('Price Label')
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('../reports/figures/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature Importance (LightGBM)

In [ ]:
fi_path = '../reports/figures/feature_importance.png'
if os.path.exists(fi_path):
    from PIL import Image
    img = Image.open(fi_path)
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title('Top Feature Importances — LightGBM')
    plt.tight_layout()
    plt.show()
else:
    print("Run train_model.py to generate feature importance plot")

    # Show correlation of features with target as proxy
    MODEL_FEATURES = ['log_asking_price','age_years','log_hours_used','total_engagement',
                      'bid_rate','inquiry_rate','condition_score','seller_sold_rate',
                      'category_freq','region_demand_proxy']
    available = [f for f in MODEL_FEATURES if f in df.columns]
    corr_with_target = df[available + ['price_label_enc']].corr()['price_label_enc'].drop('price_label_enc')

    fig, ax = plt.subplots(figsize=(8, 6))
    corr_with_target.sort_values().plot.barh(ax=ax, color=['red' if v < 0 else 'green' for v in corr_with_target.sort_values()])
    ax.set_title('Feature Correlation with Price Label')
    ax.axvline(0, color='black', linewidth=0.5)
    plt.tight_layout()
    plt.show()

## 4. Model Performance Comparison

In [ ]:
if os.path.exists(results_path):
    models = ['logistic_regression', 'lightgbm', 'xgboost']
    metrics = ['accuracy', 'precision', 'recall', 'f1_macro', 'roc_auc']

    comparison_df = pd.DataFrame({
        m_name: [results[m_name][metric] for metric in metrics]
        for m_name in models
    }, index=metrics)

    print(comparison_df.round(4))

    fig, ax = plt.subplots(figsize=(11, 5))
    comparison_df.T.plot.bar(ax=ax, width=0.7, edgecolor='white')
    ax.set_title('Model Performance Comparison')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.05)
    ax.legend(loc='lower right')
    ax.set_xticklabels([m.replace('_', '\n') for m in models], rotation=0)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('../reports/figures/model_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. Confusion Matrix

In [ ]:
cm_path = '../reports/figures/confusion_matrix.png'
if os.path.exists(cm_path):
    from PIL import Image
    img = Image.open(cm_path)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.imshow(img)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("Run train_model.py to generate confusion matrix")

## 6. Business Recommendations

1. **Underpriced listings** (≈20%): Flag these for buyers as strong value opportunities. Alert sellers to consider repricing.
2. **Overpriced listings** (≈20%): Target sellers for pricing guidance. These listings have longer days-on-market.
3. **Fair-priced listings** (≈60%): Strong candidates for featured placement; predictable demand.
4. **LightGBM** outperforms logistic regression significantly, especially on ROC-AUC — confirms non-linear pricing dynamics.
5. **Top drivers**: log price, engagement signals, condition, and category context are most predictive.

In [ ]:
print('Modeling analysis complete.')